# ARGUS · S3 — Fine-tune from S2.5 on KITTI + BDD100K + IDD

**Goal:** Fix truck mAP50 0.138 → 0.45+ by dropping overhead UAV domain (UAVDT) and adding KITTI dashcam perspective with 3× truck oversample.

**Datasets:** BDD100K (~70k) + IDD (~7k) + KITTI (~6.8k train)  
**Pretrained from:** `argus-s25.pt` (S2.5 hard-mining checkpoint)  
**Platform:** Kaggle T4 × 2  
**Training:** 45 epochs · 1280px · single phase

**Before running — add these 3 datasets as inputs:**
1. `jubayerisfar/kitti-dataset-yolo-format` (6.1 GB — KITTI pre-converted YOLO)
2. `a7madmostafa/bdd100k-yolo` (5.7 GB)
3. `redzapdos123/indian-driving-dataset-detections-yolov11` (21.9 GB)
4. Your `argus-s25-weights` dataset containing `argus-s25.pt`

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
from pathlib import Path
import torch, os, json as _json

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

NC      = 5
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

WORK     = Path('/kaggle/working')
OUT_DIR  = WORK / 'argus_data'
MERGED   = OUT_DIR / 'merged'
RUNS_DIR = WORK / 'runs'
INPUT    = Path('/kaggle/input')
yaml_path = MERGED / 'data.yaml'

for d in [MERGED/'train/images', MERGED/'train/labels',
          MERGED/'valid/images', MERGED/'valid/labels',
          RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Locate S2.5 checkpoint ───────────────────────────────────────────────────
S25_PT = WORK / 'argus_s25.pt'
if not S25_PT.exists():
    for _name in ['argus_s25_hmb_best.pt', 'argus_s25.pt', 'best.pt']:
        hit = next(INPUT.rglob(_name), None)
        if hit:
            import shutil as _sh; _sh.copy2(hit, S25_PT)
            print(f'S2.5 checkpoint: {hit} → {S25_PT}')
            break
    if not S25_PT.exists():
        # Search for any .pt with 's25' or 'hmb' in name
        for pt in sorted(INPUT.rglob('*.pt')):
            if 's25' in pt.name.lower() or 'hmb' in pt.name.lower():
                import shutil as _sh2; _sh2.copy2(pt, S25_PT)
                print(f'S2.5 checkpoint found: {pt}')
                break

if S25_PT.exists():
    print(f'✓ argus_s25.pt ready: {S25_PT.stat().st_size/1e6:.0f} MB')
else:
    print('⚠  argus_s25.pt not found.')
    print('   1. Download best.pt from Session 2.5 output')
    print('   2. Create Kaggle dataset "argus-s25-weights" with that file')
    print('   3. Add dataset to this notebook under Input')

S3_BEST = RUNS_DIR / 's3' / 'argus_s3' / 'weights' / 'best.pt'
S3_LAST = RUNS_DIR / 's3' / 'argus_s3' / 'weights' / 'last.pt'

n_gpu  = torch.cuda.device_count()
DEVICE = ','.join(str(i) for i in range(n_gpu)) or 'cpu'
BATCH  = 4 if n_gpu >= 2 else 2  # 2/GPU: YOLO12 Area Attention OOMs at 4/GPU on T4
IMGSZ  = 1024  # 1280 OOMs on T4: activation memory ∝ imgsz², 1024 fits
EPOCHS = 45

KGL_USER = 'blank0013'
KGL_KEY  = 'KGAT_20706258e7b0430b31ca3565204f2406'
if KGL_USER and KGL_KEY:
    kdir = Path.home() / '.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir / 'kaggle.json').write_text(_json.dumps({'username': KGL_USER, 'key': KGL_KEY}))
    (kdir / 'kaggle.json').chmod(0o600)
    print(f'Kaggle creds: {KGL_USER}')
else:
    print('Kaggle creds NOT set — dataset API downloads will be skipped')

print(f'DEVICE={DEVICE}  BATCH={BATCH}  IMGSZ={IMGSZ}  EPOCHS={EPOCHS}')

In [ ]:
# ── Install + GPU check ──────────────────────────────────────────────────────
import subprocess, sys, torch

n_gpu = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics>=8.4.0', 'pyyaml', 'tqdm'], check=True)
import ultralytics; ultralytics.checks()

In [ ]:
# ── Pre-flight: what's in /kaggle/input/ ─────────────────────────────────────
import os
from pathlib import Path

for item in sorted(Path('/kaggle/input').iterdir()):
    if item.is_dir():
        n_files = sum(1 for _ in item.rglob('*') if _.is_file())
        print(f'  {item.name}/ ({n_files} files)')

In [ ]:
# ── KITTI → YOLO ─────────────────────────────────────────────────────────────
# Source: jubayerisfar/kitti-dataset-yolo-format (Kaggle input, 6.1 GB)
# Images + labels co-located in same directory (no images/ labels/ split)
# Car/Van→car(0)  Truck→truck(3) ×3 oversample  Tram→bus(2)  Cyclist→bicycle(4)
import os, random
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

FLAG = OUT_DIR / '.kitti_done'
if FLAG.exists():
    print('KITTI: already done — skipping')
else:
    KITTI_BASE = INPUT / 'kitti-dataset-yolo-format' / 'Kitti Dataset Yolo Format'

    if not KITTI_BASE.exists():
        print('⚠  KITTI dataset not found in /kaggle/input/')
        print('   Add "jubayerisfar/kitti-dataset-yolo-format" as a Kaggle input dataset')
        FLAG.touch()
    else:
        # jubayerisfar class IDs → ARGUS IDs
        # Car(0) Van(2) → car(0) | Cyclist(3) → bicycle(4) | Truck(4) → truck(3) | Tram(6) → bus(2)
        KMAP = {0: 0, 2: 0, 3: 4, 4: 3, 6: 2}
        TRUCK_OVERSAMPLE = 3

        for src_split, dst_split in [('train', 'train'), ('valid', 'valid')]:
            src_dir = KITTI_BASE / src_split
            if not src_dir.exists():
                print(f'  KITTI {src_split}: not found — skip'); continue

            imgs = sorted(src_dir.glob('*.png'))
            di = MERGED / dst_split / 'images'
            dl = MERGED / dst_split / 'labels'
            n_imgs = n_trucks = 0

            for img in tqdm(imgs, desc=f'KITTI {src_split}'):
                lp = src_dir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines, has_truck = [], False
                for row in lp.read_text().strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in KMAP:
                        ac = KMAP[orig]
                        lines.append(f'{ac} ' + ' '.join(p[1:]))
                        if ac == 3: has_truck = True
                if not lines: continue
                stem = f'kitti_{img.stem}'
                if not (di / (stem + '.png')).exists(): _imglink(img, di / (stem + '.png'))
                (dl / (stem + '.txt')).write_text('\n'.join(lines))
                n_imgs += 1
                if has_truck and src_split == 'train':
                    n_trucks += 1
                    for k in range(1, TRUCK_OVERSAMPLE):
                        s2 = f'kitti_{img.stem}_t{k}'
                        if not (di / (s2 + '.png')).exists(): _imglink(img, di / (s2 + '.png'))
                        (dl / (s2 + '.txt')).write_text('\n'.join(lines))

            print(f'  KITTI {src_split}: {n_imgs} imgs | truck×{TRUCK_OVERSAMPLE}: {n_trucks}')

        FLAG.touch()
        print('KITTI: done (dataset lives in /kaggle/input/ — zero working disk used)')

In [ ]:
# ── BDD100K → YOLO ───────────────────────────────────────────────────────────
# motorcycle ×3 oversample, bicycle ×2 oversample for class balance.
# Checks /kaggle/input/ first (attached dataset), falls back to Kaggle API.
import shutil as _shutil

def _imglink(src: 'Path', dst: 'Path'):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)
import os, subprocess, yaml
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.bdd_done'
if FLAG.exists():
    print('BDD100K: already done — skipping')
else:
    name_to_argus = {'car':0,'automobile':0,'motorcycle':1,'motorbike':1,'motor':1,
                     'bus':2,'truck':3,'van':3,'bicycle':4,'bike':4}

    BDD_DIR = None

    # 1. Check /kaggle/input/ for an attached dataset
    ATTACHED_SLUGS = ['bdd100k-yolo-format','bdd100k-dataset','bdd100k','bdd100k-yolo',
                      'bdd100k-yolo-v8','bdd-100k','bdd100k_yolo','bdd_100k']
    for slug in ATTACHED_SLUGS:
        candidate = INPUT / slug
        if candidate.exists() and any(candidate.rglob('*.jpg')):
            BDD_DIR = candidate
            print(f'BDD100K: found attached dataset at {BDD_DIR}')
            break

    # 2. Search nested datasets/<user>/<ds>/ (Kaggle merged dataset mounting)
    if BDD_DIR is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or BDD_DIR: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    for yf in ds_dir.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(yf.read_text())
                            names_list = cfg.get('names', [])
                            if isinstance(names_list, dict):
                                names_list = list(names_list.values())
                            bdd_markers = {'rider', 'traffic light', 'traffic sign', 'motor'}
                            if bdd_markers & {n.strip().lower() for n in names_list}:
                                BDD_DIR = ds_dir
                                print(f'BDD100K: found at {ds_dir.relative_to(INPUT)}')
                                break
                        except Exception: pass
                    if BDD_DIR: break

    # 3. Fallback: API download
    if BDD_DIR is None and KGL_USER:
        RAW = WORK / '_bdd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        API_SLUGS = ['a7madmostafa/bdd100k-yolo',
                     'farzadnekouei/bdd100k-yolo-format','awsaf49/bdd100k-dataset',
                     'a2zcode/bdd100k']
        if dl_flag.exists():
            print('BDD100K: already downloaded'); BDD_DIR = RAW
        else:
            for slug in API_SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); BDD_DIR = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:120]}')

    if BDD_DIR is None:
        print('BDD100K: not found — attach dataset or set Kaggle Secrets')
    else:
        bdd_map = None
        for yf in sorted(BDD_DIR.rglob('*.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                m = {i: name_to_argus[n.lower().strip()] for i, n in enumerate(names)
                     if n.lower().strip() in name_to_argus}
                if m: bdd_map = m; print(f'  BDD map: {m}'); break
            except Exception: pass
        if bdd_map is None:
            # BDD100K 10-class: pedestrian(0) rider(1) car(2) truck(3) bus(4)
            #                   train(5) motorcycle(6) bicycle(7) tl(8) ts(9)
            bdd_map = {2:0, 6:1, 3:2, 4:3, 5:4}
            print(f'  BDD map: fallback {bdd_map}')

        for split, dst in [('train', 'train'), ('val', 'valid')]:
            idirs = [d for d in BDD_DIR.rglob('images') if split in str(d)]
            ldirs = [d for d in BDD_DIR.rglob('labels') if split in str(d)]
            if not idirs: print(f'  BDD {split}: not found — skip'); continue
            idir = idirs[0]
            ldir = ldirs[0] if ldirs else idir.parent.parent / 'labels' / split
            di = MERGED / dst / 'images'; dl = MERGED / dst / 'labels'
            nv = nm = nb_cnt = 0
            for img in tqdm(list(idir.glob('*.*')), desc=f'BDD {split}'):
                lp = ldir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines, has_m, has_b = [], False, False
                for row in lp.read_text().strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in bdd_map:
                        ac = bdd_map[orig]; lines.append(f'{ac} ' + ' '.join(p[1:]))
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                if not lines: continue
                stem = f'bdd_{img.stem}'
                link = di / (stem + img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl / (stem + '.txt')).write_text('\n'.join(lines))
                nv += 1
                if has_m:
                    nm += 1
                    for k in range(2):
                        s2 = f'bdd_{img.stem}_m{k}'
                        ml = di / (s2 + img.suffix)
                        if not ml.exists(): _imglink(img, ml)
                        (dl / (s2 + '.txt')).write_text('\n'.join(lines))
                if has_b:
                    nb_cnt += 1
                    bl = di / (f'bdd_{img.stem}_b0' + img.suffix)
                    if not bl.exists(): _imglink(img, bl)
                    (dl / (f'bdd_{img.stem}_b0.txt')).write_text('\n'.join(lines))
            print(f'  BDD {split}: {nv} imgs | moto×3: {nm} | bike×2: {nb_cnt}')
        FLAG.touch(); print('BDD100K: done')
        if str(BDD_DIR).startswith(str(WORK)):
            _shutil.rmtree(BDD_DIR, ignore_errors=True)

In [ ]:
# ── IDD (Indian Driving Dataset) → YOLO ──────────────────────────────────────
# motorcycle/autorickshaw ×4, bicycle ×2  (high-density Indian traffic)
import os, subprocess, yaml, shutil as _shutil
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

FLAG = OUT_DIR / '.idd_done'
if FLAG.exists():
    print('IDD: already done — skipping')
else:
    idd_name_map = {
        'car':0,'suv':0,'sedan':0,'hatchback':0,'mpv':0,
        'motorcycle':1,'motorbike':1,'scooter':1,'autorickshaw':1,
        'bus':2,'truck':3,'vehicle fallback':3,
        'bicycle':4,'cyclist':4,
    }

    IDD_DIR = None

    ATTACHED_SLUGS = ['indian-driving-dataset-detections-yolov11',
                      'indian-driving-dataset-idd-for-object-detection',
                      'idd-object-detection','idd','idd20k','indian-driving-dataset']
    for slug in ATTACHED_SLUGS:
        c = INPUT / slug
        if c.exists() and any(c.rglob('*.jpg')):
            IDD_DIR = c; print(f'IDD: found at {IDD_DIR}'); break

    if IDD_DIR is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or IDD_DIR: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    for yf in ds_dir.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(yf.read_text())
                            names = cfg.get('names', [])
                            if isinstance(names, dict): names = list(names.values())
                            idd_markers = {'autorickshaw', 'auto', 'rickshaw'}
                            if idd_markers & {n.strip().lower() for n in names}:
                                IDD_DIR = ds_dir
                                print(f'IDD: found at {ds_dir.relative_to(INPUT)}')
                                break
                        except Exception: pass
                    if IDD_DIR: break

    if IDD_DIR is None and KGL_USER:
        RAW = WORK / '_idd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        API_SLUGS = ['redzapdos123/indian-driving-dataset-detections-yolov11',
                     'brdorlando/indian-driving-dataset-idd-for-object-detection',
                     'noorulhaq/idd-object-detection-v2']
        if dl_flag.exists():
            print('IDD: already downloaded'); IDD_DIR = RAW
        else:
            for slug in API_SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); IDD_DIR = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:120]}')

    if IDD_DIR is None:
        print('IDD: not found — attach dataset or set Kaggle Secrets')
    else:
        idd_map = None
        for yf in sorted(IDD_DIR.rglob('*.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                m = {i: idd_name_map[n.lower().strip()] for i, n in enumerate(names)
                     if n.lower().strip() in idd_name_map}
                if m: idd_map = m; print(f'  IDD map: {m}'); break
            except Exception: pass

        if idd_map is None:
            print('IDD: could not detect class map from data.yaml — skipping')
            FLAG.touch()
        else:
            for split, dst in [('train', 'train'), ('val', 'valid')]:
                idirs = [d for d in IDD_DIR.rglob('images') if split in str(d)]
                ldirs = [d for d in IDD_DIR.rglob('labels') if split in str(d)]
                if not idirs: print(f'  IDD {split}: not found — skip'); continue
                idir = idirs[0]
                ldir = ldirs[0] if ldirs else idir.parent.parent / 'labels' / split
                di = MERGED / dst / 'images'; dl = MERGED / dst / 'labels'
                nv = nm = nb_cnt = 0
                for img in tqdm(list(idir.glob('*.*')), desc=f'IDD {split}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]; lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): _imglink(img, link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            s2 = f'idd_{img.stem}_m{k}'
                            ml = di / (s2 + img.suffix)
                            if not ml.exists(): _imglink(img, ml)
                            (dl / (s2 + '.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb_cnt += 1
                        bl = di / (f'idd_{img.stem}_b0' + img.suffix)
                        if not bl.exists(): _imglink(img, bl)
                        (dl / (f'idd_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD {split}: {nv} imgs | moto×4: {nm} | bike×2: {nb_cnt}')
            FLAG.touch(); print('IDD: done')
        if str(IDD_DIR).startswith(str(WORK)):
            _shutil.rmtree(IDD_DIR, ignore_errors=True)

In [ ]:
# ── Dataset stats + data.yaml ─────────────────────────────────────────────────
from pathlib import Path
from collections import Counter

def _count(split):
    ldir = MERGED / split / 'labels'
    imgs = len(list((MERGED / split / 'images').glob('*.*')))
    stats = Counter()
    for lp in ldir.glob('*.txt'):
        for row in lp.read_text().splitlines():
            p = row.strip().split()
            if p:
                try: stats[int(p[0])] += 1
                except ValueError: pass
    return imgs, stats

print('=' * 60)
for split in ['train', 'valid']:
    n, stats = _count(split)
    total = sum(stats.values())
    bad = {k: v for k, v in stats.items() if k < 0 or k >= NC}
    print(f'\n{split}: {n:,} images | {total:,} boxes')
    for cid, name in enumerate(CLASSES):
        bar = '█' * min(40, int(40 * stats.get(cid, 0) / max(total, 1)))
        print(f'  {cid} {name:12s}: {stats.get(cid, 0):8,}  {bar}')
    if bad:
        raise ValueError(f'OUT-OF-RANGE class IDs in {split}: {bad}')
    print(f'  ✓ all IDs in [0,{NC-1}]')
print('=' * 60)

n_train = len(list((MERGED / 'train' / 'images').glob('*.*')))
n_moto  = sum(1 for lp in (MERGED / 'train' / 'labels').glob('*.txt')
              if any(r.split()[0] == '1' for r in lp.read_text().splitlines() if r.strip()))
n_truck = sum(1 for lp in (MERGED / 'train' / 'labels').glob('*.txt')
              if any(r.split()[0] == '3' for r in lp.read_text().splitlines() if r.strip()))
assert n_train > 10_000, f'Only {n_train} training images — check dataset downloads'
assert n_moto  > 500,   f'Only {n_moto} motorcycle images — BDD class map likely broken'
assert n_truck > 500,   f'Only {n_truck} truck images — KITTI download may have failed'

yaml_path.write_text(
    f'path: {MERGED.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n'
)
print(f'\ndata.yaml → {yaml_path}')
print(yaml_path.read_text())

In [ ]:
# ── Train 45 epochs from S2.5 checkpoint ─────────────────────────────────────
# Fine-tuning: low LR (5e-5), 1280px, close_mosaic=10 (clean-image final phase).
# Expected: truck mAP50 0.138 → 0.45+  (UAVDT overhead domain removed)
from ultralytics import YOLO
import shutil, time

assert yaml_path.exists(), 'Run dataset cells first'

if not S25_PT.exists():
    raise FileNotFoundError(
        'argus_s25.pt not found.\n'
        'Upload argus_s25_hmb_best.pt as Kaggle dataset "argus-s25-weights" and add to Input.'
    )

# YOLO12 Area Attention has conditionally-unused parameters; patch DDP before training
# ultralytics 8.3-8.4.x uses DDP(self.model, device_ids=[RANK]) with no explicit
# find_unused_parameters — defaults to False, which breaks with Area Attention.
import pathlib as _pl, re as _re
import ultralytics.engine.trainer as _tr
_tr_file = _pl.Path(_tr.__file__)
_src = _tr_file.read_text()

# Print DDP-related lines so any future failure is instantly diagnosable
print("DDP lines in trainer.py:")
for _ln in _src.splitlines():
    if "DDP" in _ln or "find_unused" in _ln:
        print(" ", repr(_ln))

_new = _src
if "find_unused_parameters=False" in _src:
    _new = _src.replace("find_unused_parameters=False", "find_unused_parameters=True")
    print("DDP patch A: False→True")
elif _re.search(r"DDP\(self\.model,\s*device_ids=\[RANK\]\s*\)", _src):
    # Bare DDP call — ultralytics 8.4.x omits find_unused_parameters (defaults False)
    _new = _re.sub(
        r"DDP\(self\.model,\s*device_ids=\[RANK\]\s*\)",
        "DDP(self.model, device_ids=[RANK], find_unused_parameters=True)",
        _src)
    print("DDP patch B: injected find_unused_parameters=True")
elif "DDP(self.model" in _src:
    # Generic: inject into any single-line DDP(self.model...) call
    _new = _re.sub(r"(DDP\(self\.model[^\n)]*)\)", r"\1, find_unused_parameters=True)", _src)
    if _new != _src:
        print("DDP patch C: generic injection")
    else:
        print("WARNING: all DDP patches failed")

if _new != _src:
    _tr_file.write_text(_new)
    _cleared = []
    _pyc_dir = _tr_file.parent / "__pycache__"
    for _pyc in _pyc_dir.glob("trainer.cpython-*.pyc"):
        _pyc.unlink(missing_ok=True)
        _cleared.append(_pyc.name)
    print(f"trainer.py patched, .pyc cleared: {_cleared}")
else:
    print("PATCH FAILED — no DDP pattern matched; check printed lines above")
    print("Fallback: change DEVICE to 0 (single GPU) to avoid DDP")

t0 = time.time()
if S3_LAST.exists():
    print(f'Resuming from checkpoint: {S3_LAST}')
    model = YOLO(str(S3_LAST))
    model.train(resume=True)
else:
    print(f'Starting from S2.5: {S25_PT}')
    model = YOLO(str(S25_PT))
    model.train(
        data=str(yaml_path),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        project=str(RUNS_DIR),
        name='argus_s3',
        # Fine-tuning LR: start low since S2.5 is already well-converged
        optimizer='SGD',  # prevent optimizer=auto overriding lr0
        lr0=5e-5,
        lrf=0.1,
        warmup_epochs=3,
        weight_decay=5e-4,
        # Loss weights carried from S2.5
        box=9.0, cls=0.3, dfl=2.0,
        # Augmentation: moderate — fine-tuning not training from scratch
        mosaic=0.9,
        close_mosaic=10,   # last 10 epochs: clean images, mimics inference
        mixup=0.05,
        copy_paste=0.05,
        erasing=0.1,
        fliplr=0.5,
        scale=0.4,
        degrees=5.0,
        patience=15,
        save_period=5,
        amp=True,
        exist_ok=True,
    )

elapsed = (time.time() - t0) / 3600
print(f'\nTraining done in {elapsed:.1f} h')

if S3_BEST.exists():
    shutil.copy2(S3_BEST, WORK / 'argus_s3.pt')
    print(f'✓ Best weights → {WORK}/argus_s3.pt')

In [ ]:
# ── Validation + per-class results ───────────────────────────────────────────
# conf=0.001: exposes all detections (standard COCO eval practice).
# iou=0.6:    stricter box match — penalises imprecise localisation for TTC.
from ultralytics import YOLO

candidates = [WORK / 'argus_s3.pt', S3_BEST, S25_PT]
EVAL_MODEL = next((c for c in candidates if c.exists()), None)
assert EVAL_MODEL, 'No trained model found — run training cell first'
print(f'Evaluating: {EVAL_MODEL}')

model   = YOLO(str(EVAL_MODEL))
metrics = model.val(
    data=str(yaml_path), imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    conf=0.001, iou=0.6, verbose=True,
)

print('\n' + '=' * 60)
print('  ARGUS S3 Validation Results')
print('=' * 60)
print(f'  mAP50    : {metrics.box.map50:.4f}')
print(f'  mAP50-95 : {metrics.box.map:.4f}  ← primary metric')
print()

# Use ap_class_index for correct per-class alignment
# (ap50 / maps only include classes present in val set)
names_by_id = model.names
for cls_idx, ap50, ap in zip(metrics.box.ap_class_index,
                              metrics.box.ap50,
                              metrics.box.maps):
    name = names_by_id.get(int(cls_idx), str(cls_idx))
    flag = '  ← TARGET (was 0.138 on S2.5)' if name == 'truck' else ''
    print(f'  {name:12s}: AP50={ap50:.4f}  AP50-95={ap:.4f}{flag}')

print('=' * 60)

# Get truck index
truck_idx = next((i for i, c in enumerate(metrics.box.ap_class_index)
                  if names_by_id.get(int(c)) == 'truck'), None)
if truck_idx is not None:
    truck50 = metrics.box.ap50[truck_idx]
    if truck50 < 0.35:   print(f'\n⚠  Truck AP50={truck50:.3f} < 0.35 — KITTI cell may have failed')
    elif truck50 < 0.45: print(f'\n→  Truck AP50={truck50:.3f} — improved but below target')
    else:                print(f'\n✓  Truck AP50={truck50:.3f} — target reached')